# Day 1: Setup and Smoketest
This notebook covers Kaggle dataset setup, eval-driven species curation, and a smoke test of BirdNET, Perch, and AVES.

In [ ]:
!pip install -q birdnetlib transformers audiomentations

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Setup paths dynamically by searching /kaggle/input
INPUT_DIR = Path('/kaggle/input')
DATA_OUT = Path('./data')
DATA_OUT.mkdir(exist_ok=True)

# Find the files automatically
try:
    TRAIN_CSV = next(INPUT_DIR.rglob('train_metadata.csv'))
    print(f"Found Train CSV: {TRAIN_CSV}")
except StopIteration:
    TRAIN_CSV = Path('not_found')

eval_labels_paths = [p for p in INPUT_DIR.rglob('*.csv') if 'train_metadata' not in p.name and 'test' not in p.name]
EVAL_LABELS_CSV = eval_labels_paths[0] if eval_labels_paths else Path('not_found')
if EVAL_LABELS_CSV.exists():
    print(f"Found Eval CSV: {EVAL_LABELS_CSV}")

# 1. Eval-driven curation
focus_from_eval = []
if EVAL_LABELS_CSV.exists():
    eval_df = pd.read_csv(EVAL_LABELS_CSV)
    print(f'Eval dataset shape: {eval_df.shape}')
    
    # Get species columns (excluding row_id, audio_id, etc)
    species_cols = [c for c in eval_df.columns if c not in ['row_id', 'audio_id', 'time', 'site']]
    eval_counts = eval_df[species_cols].sum().sort_values(ascending=False)
    focus_from_eval = eval_counts[eval_counts > 0].index.tolist()
    print(f'Found {len(focus_from_eval)} species with >=1 positive in eval.')
    
    eval_counts.to_frame('eval_positives').to_csv(DATA_OUT / 'eval_coverage.csv')
else:
    print('Eval dataset not found.')

# 2. Focus Species Generation
if TRAIN_CSV.exists():
    train_df = pd.read_csv(TRAIN_CSV)
    print(f'Train metadata shape: {train_df.shape}')
    
    train_counts = train_df['primary_label'].value_counts()
    focus_set = set(focus_from_eval)
    
    # Pad up to 50 using the most abundant training species
    target_len = 50
    for sp in train_counts.index:
        if len(focus_set) >= target_len:
            break
        focus_set.add(sp)
        
    focus_list = sorted(list(focus_set))
    pd.DataFrame({'species': focus_list}).to_csv(DATA_OUT / 'species_focus.csv', index=False)
    print(f'Selected {len(focus_list)} focus species and saved to data/species_focus.csv')
else:
    print('Train metadata not found.')

In [ ]:
%%writefile extract_embeddings.py
import librosa
import numpy as np
import tensorflow as tf
from huggingface_hub import snapshot_download
from birdnetlib import Recording
from birdnetlib.analyzer import Analyzer
from transformers import AutoModel, AutoFeatureExtractor
import torch

SR = 32000
CHUNK_SEC = 5.0

def load_audio(path: str, sr: int = SR) -> np.ndarray:
    try:
        y, _ = librosa.load(path, sr=sr, mono=True)
        return y
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return np.array([])

# BirdNET
try:
    analyzer = Analyzer()
except Exception as e:
    analyzer = None

def birdnet_predict(audio_path: str, lat=10.5, lon=76.5, week=-1):
    if not analyzer: return []
    rec = Recording(analyzer, audio_path, lat=lat, lon=lon, week=week, min_conf=0.0, overlap=0.0)
    rec.analyze()
    return rec.detections

# Perch
try:
    PERCH_REPO = "cgeorgiaw/Perch"
    local_perch = snapshot_download(repo_id=PERCH_REPO)
    perch = tf.saved_model.load(local_perch)
except Exception as e:
    perch = None

def perch_embed(audio_5s_32k: np.ndarray) -> np.ndarray:
    if not perch: return np.zeros(1536)
    x = tf.constant(audio_5s_32k[None, :], dtype=tf.float32)
    out = perch.infer_tf(x)
    return out["embedding"].numpy()[0]

# AVES
try:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    aves = AutoModel.from_pretrained("earthspecies/aves-base-bio").eval().to(device)
    aves_fx = AutoFeatureExtractor.from_pretrained("earthspecies/aves-base-bio")
except Exception as e:
    aves = None

@torch.no_grad()
def aves_embed(audio_5s_16k: np.ndarray) -> np.ndarray:
    if not aves: return np.zeros(768)
    x = aves_fx(audio_5s_16k, sampling_rate=16000, return_tensors="pt").input_values.to(device)
    h = aves(x).last_hidden_state.mean(dim=1)
    return h.cpu().numpy()[0]

In [ ]:
import extract_embeddings
print("Models loaded successfully! Smoke test passed.")